In [ ]:
import sys
sys.path.append('/home/unix/vkrhk/EmmaEmb/EmmaEmb/analysis/')
from dim_reduction_utils import load_dataset_with_all_balanced_classes, load_imbalanced_cryptic_and_regular_data, mean_center
import constants
from emmaemb.vizualisation import get_knn_alignment_scores

ANNOY_TREES = 100
METRIC = 'cosine'
ANNOY_METRIC = METRIC
K = 100

emma = load_dataset_with_all_balanced_classes()
mean_center(emma, emb_spaces=['ESM2', 'ANKH', 'ProstT5', 'ProtT5'])
for embeddings_name, _ in constants.EMB_SPACES:
    emma.build_annoy_index(emb_space=embeddings_name, metric=ANNOY_METRIC, n_trees=ANNOY_TREES)
df = get_knn_alignment_scores(emma, feature='binding_site', k=100, metric=METRIC, use_annoy=True, n_trees=ANNOY_TREES, annoy_metric=ANNOY_METRIC)


In [18]:
import pingouin as pg

results = []
tests = [('ProstT5', 'ProtT5'), ('ProtT5', 'ANKH'), ('ANKH', 'ESM2')]
for better, worse in tests:
    x = df[(df["Embedding"] == better)]['Fraction'].to_numpy()
    y = df[(df["Embedding"] == worse)]['Fraction'].to_numpy()
    stats = pg.wilcoxon(x, y, alternative="greater")
    results.append({
                'metric': "cosine",
                'better': better,
                'worse': worse,
                'p_value': stats['p_val'].values[0],
                'rank_biserial': stats['RBC'].values[0]
            })
results

[{'metric': 'cosine',
  'better': 'ProstT5',
  'worse': 'ProtT5',
  'p_value': np.float64(1.6171469756805795e-49),
  'rank_biserial': np.float64(0.08805070226260975)},
 {'metric': 'cosine',
  'better': 'ProtT5',
  'worse': 'ANKH',
  'p_value': np.float64(1.8077962875472993e-68),
  'rank_biserial': np.float64(0.10438053837611455)},
 {'metric': 'cosine',
  'better': 'ANKH',
  'worse': 'ESM2',
  'p_value': np.float64(9.18879057756832e-182),
  'rank_biserial': np.float64(0.17081620242587592)}]

In [19]:
import pandas as pd

df_results = pd.DataFrame(results)
# write into file
output_dir = '/home/unix/vkrhk/EmmaEmb/EmmaEmb/analysis/figs'
df_results.to_csv(output_dir + "/wilcoxon.csv", index=False)